In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI,OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate


In [14]:
FAISS_PATH="faiss_index"
import os

def create_faiss_index(file_path):

    if os.path.exists(FAISS_PATH):
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        print("Loading FAISS locally ...")
        return FAISS.load_local(FAISS_PATH, embeddings,allow_dangerous_deserialization=True)
    
    else:
        print("creating FAISS index...")

        text_loader = TextLoader(file_path)
        documents = text_loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        chunks= text_splitter.split_documents(documents)
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        vectorstore = FAISS.from_documents(chunks, embeddings)
        vectorstore.save_local(FAISS_PATH)
        print("FAISS index created and saved locally.")
        return vectorstore



In [15]:
FAISS_vectorstore=create_faiss_index("ml.txt")

creating FAISS index...
FAISS index created and saved locally.


In [17]:
retriever = FAISS_vectorstore.as_retriever()

docs=retriever.invoke("What is linear regression?",k=5)


for i,doc in enumerate(docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Document 1:
Linear Regression performs well when relationships between variables are approximately linear. It is commonly used for house price prediction, sales forecasting, demand prediction, and trend analysis.

One major advantage of Linear Regression is interpretability. The coefficients explain how strongly each feature affects the output prediction.

Document 2:
Regression algorithms predict continuous numerical values such as house prices, stock prices, temperature values, and sales forecasts.

Linear Regression is one of the simplest and most interpretable machine learning algorithms. It models the relationship between input features and a continuous target variable using a linear equation.

Document 3:
Linear Regression is computationally efficient and easy to train even on large datasets. However, it struggles with highly nonlinear relationships and is sensitive to outliers.

Polynomial Regression extends Linear Regression by introducing polynomial features to model curved re

In [19]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

In [23]:
text_loader = TextLoader("ml.txt")
documents = text_loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks= text_splitter.split_documents(documents)
sparse_retriever = BM25Retriever.from_documents(chunks, k=5)

In [24]:
hybrid_retriever = EnsembleRetriever(retrievers=[retriever, sparse_retriever], weights=[0.7, 0.3])

In [25]:
docs=hybrid_retriever.invoke("What is linear regression?",k=5)

for i,doc in enumerate(docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Document 1:
Regression algorithms predict continuous numerical values such as house prices, stock prices, temperature values, and sales forecasts.

Linear Regression is one of the simplest and most interpretable machine learning algorithms. It models the relationship between input features and a continuous target variable using a linear equation.

Document 2:
Logistic Regression is used for classification tasks despite having the term regression in its name. It predicts probabilities and is commonly used in binary classification problems such as spam detection and fraud prediction.

Logistic Regression is highly interpretable and computationally efficient. It works well when decision boundaries are approximately linear.

Document 3:
Linear Regression performs well when relationships between variables are approximately linear. It is commonly used for house price prediction, sales forecasting, demand prediction, and trend analysis.

One major advantage of Linear Regression is interpretab

In [32]:
dense_retriever = FAISS_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)

sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 5

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

docs=hybrid_retriever.invoke("What is linear regression?",k=5)

for i,doc in enumerate(docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Document 1:
Regression algorithms predict continuous numerical values such as house prices, stock prices, temperature values, and sales forecasts.

Linear Regression is one of the simplest and most interpretable machine learning algorithms. It models the relationship between input features and a continuous target variable using a linear equation.

Document 2:
Logistic Regression is used for classification tasks despite having the term regression in its name. It predicts probabilities and is commonly used in binary classification problems such as spam detection and fraud prediction.

Logistic Regression is highly interpretable and computationally efficient. It works well when decision boundaries are approximately linear.

Document 3:
Linear Regression is suitable when relationships are approximately linear and interpretability is important.

Logistic Regression is suitable for simple classification tasks requiring explainability.

Decision Trees are useful when interpretability and nonl

In [33]:
dense_retriever_2=FAISS_vectorstore.as_retriever(k=5)

sparse_retriever_2 = BM25Retriever.from_documents(chunks, k=5)


hybrid_retriever_2 = EnsembleRetriever(
    retrievers=[dense_retriever_2, sparse_retriever_2], 
    weights=[0.7, 0.3]
)



In [35]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

base_retriever=hybrid_retriever_2
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)
docs = compression_retriever.invoke("What is linear regression?", k=5)
for i, doc in enumerate(docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 951.24it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Document 1:
Regression algorithms predict continuous numerical values such as house prices, stock prices, temperature values, and sales forecasts.

Linear Regression is one of the simplest and most interpretable machine learning algorithms. It models the relationship between input features and a continuous target variable using a linear equation.

Document 2:
Linear Regression is computationally efficient and easy to train even on large datasets. However, it struggles with highly nonlinear relationships and is sensitive to outliers.

Polynomial Regression extends Linear Regression by introducing polynomial features to model curved relationships.

Document 3:
Linear Regression performs well when relationships between variables are approximately linear. It is commonly used for house price prediction, sales forecasting, demand prediction, and trend analysis.

One major advantage of Linear Regression is interpretability. The coefficients explain how strongly each feature affects the output